# maxpool-reduce — ex1: build MaxPool2d via einops.reduce

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `maxpool-reduce`. Running the final beacon cell reports progress against the `CNN: MaxPool as reduce` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: MaxPool as reduce` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`maxpool-reduce`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "maxpool-reduce"
DD_SUBTOPIC = "CNN: MaxPool as reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## MaxPool2d == einops.reduce(max) — quick refresher

`nn.MaxPool2d(p)` with non-overlapping windows is exactly an einops `reduce` with the `max` op:

```
y = einops.reduce(x, 'b c (h p1) (w p2) -> b c h w', 'max', p1=p, p2=p)
```

**Reading the einops string** (same as AvgPool — only the op differs):
- `(h p1)` factors the H-axis into `h * p1` — `h` is the output spatial axis, `p1` is the pool-window axis being reduced.
- `'max'` takes the maximum value across the dropped axes (vs `'mean'` for AvgPool).

**Why MaxPool is the canonical CNN pool.** It introduces translation robustness — small shifts in the input don't change the max-of-window if the activating pixel stays inside the window. AlexNet, VGG, and ResNet's stem all use MaxPool right after the first conv.

**Why ResNet uses Max in the stem but Avg at the head.** The stem's MaxPool selects salient activations early. The head's global AvgPool (after the last BlockGroup) averages across the whole feature map to produce one scalar per channel — that smoothing is what a classifier wants. Max would be too noisy for the final pool.

**Compared to AvgPool.** Identical einops pattern; identical shape math. The op `'max'` vs `'mean'` is the ONLY difference. Same KC (pool-as-einops-reduce) — drilling Max separately reinforces that the axis-factoring trick is op-agnostic.

### Exercise 1 — build MaxPool2d via einops.reduce

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `einops.reduce('max')` with axis-factoring to reproduce `nn.MaxPool2d` with non-overlapping windows, and verify against `F.max_pool2d`.
> Keywords: maxpool, einops-reduce, translation-invariance, resnet-stem
> ```

**KCs targeted:** `maxpool-as-reduce-max`, `pool-axis-factor-pattern`

Implement `ex1_maxpool_via_reduce(x, p)`. Given input `x: (B, C, H, W)` and pool size `p` (assume `H` and `W` are divisible by `p`), return a `(B, C, H // p, W // p)` tensor whose entries are the **max** of each non-overlapping `p × p` window of `x`.

**Use einops.reduce with axis factoring.** The pattern is:

```
einops.reduce(x, 'b c (h p1) (w p2) -> b c h w', 'max', p1=p, p2=p)
```

**Read it letter-by-letter** (same axis-factor pattern as the avgpool-reduce drill — only the reducer differs):
- `(h p1)` factors the input H-axis into `h * p1`.
- `'max'` takes the max across the dropped `p1, p2` axes.
- Pass `p1=p, p2=p` so einops knows the factor sizes.

**Boundary handling.** This drill assumes `H % p == 0` and `W % p == 0`. Real `nn.MaxPool2d` can also handle stride != kernel and padding — out of scope here.

The test compares your output to `F.max_pool2d(x, kernel_size=p)` to fp tolerance.

In [ ]:
def ex1_maxpool_via_reduce(x: Tensor, p: int) -> Tensor:
    return einops.reduce(
        x,
        'b c (h p1) (w p2) -> b c h w',
        'max',
        p1=p, p2=p,
    )


<details><summary>Solution</summary>

```python
def ex1_maxpool_via_reduce(x: Tensor, p: int) -> Tensor:
    return einops.reduce(
        x,
        'b c (h p1) (w p2) -> b c h w',
        'max',
        p1=p, p2=p,
    )
```

**Same einops pattern as AvgPool, different reducer.** The axis-factor trick `(h p1)` works for ANY commutative-monoid reduction — `'mean'`, `'max'`, `'min'`, `'sum'`, `'prod'`. Recognizing pool-as-reduce gives you N-dim pooling with arbitrary reducers for free.

**Why MaxPool is in the ResNet stem.** After the 7×7 stride-2 conv, the stem applies `MaxPool2d(3, stride=2, padding=1)` — a 3×3 OVERLAPPING max-pool (stride < kernel). This drill is the non-overlapping `kernel == stride` case; the stem's overlapping variant uses the same op, different einops doesn't directly handle (overlap needs unfold + reduce).

**Backward-pass subtlety.** Unlike AvgPool (every input contributes 1/p² to the output gradient), MaxPool routes the ENTIRE output gradient to the SINGLE max-position input — a sparse, switch-like backward. `einops.reduce(..., 'max')` handles this correctly via PyTorch's autograd, but if you ever roll your own pool, remember the sparsity.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()